# ETL Pipeline — House Price Prediction (Ames Housing Dataset)

This notebook consolidates the full data pipeline across all project notebooks:

| Phase | Source Notebook |
|-------|----------------|
| **Extract** | `01_extraction.ipynb` |
| **Transform** | `02_cleaning.ipynb` + `03_eda.ipynb` |
| **Load** | `05_final_load_prep.ipynb` |

**Outputs:**
- `data/processed/cleaned_data.csv` — fully cleaned & engineered dataset (291 cols)
- `data/processed/tableau_ready.csv` — Tableau-ready dataset with `PriceCategory` column

---
## 0. Setup — Imports & Configuration

In [ ]:
!pip install numpy pandas --quiet

In [ ]:
import os
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda x: f'{x:.3f}')

# ── Paths (relative to this notebook in scripts/) ───────────────────────────
RAW_PATH     = '../data/raw/house_price.csv'
CLEAN_PATH   = '../data/processed/cleaned_data.csv'
TABLEAU_PATH = '../data/processed/tableau_ready.csv'

# ── Pipeline constants ───────────────────────────────────────────────────────
LUXURY_PERCENTILE  = 0.99          # top 1 % → is_luxury flag
NEAR_CONST_THRESH  = 0.99          # ≥ 99 % same value → drop column
CRISIS_YEARS       = {2007, 2008, 2009, 2010}

print('✅  Setup complete')

---
## 1. EXTRACT — Load Raw Data

In [ ]:
# Load raw dataset
df = pd.read_csv(RAW_PATH)

print('✅  Raw dataset loaded')
print(f'   File    : {RAW_PATH}')
print(f'   Rows    : {df.shape[0]:,}')
print(f'   Columns : {df.shape[1]:,}')

In [ ]:
# Preview first 5 rows
print('First 5 rows:')
df.head()

In [ ]:
# ── Baseline data audit ──────────────────────────────────────────────────────
total_missing  = df.isnull().sum().sum()
dup_rows       = df.duplicated().sum()
dup_ids        = df['Id'].duplicated().sum()
const_cols     = [c for c in df.columns if df[c].nunique() <= 1]
near_const_cols = [
    c for c in df.columns
    if df[c].value_counts(normalize=True).iloc[0] >= NEAR_CONST_THRESH
]

print('=' * 55)
print('BASELINE AUDIT')
print('=' * 55)
print(f'\n[1] Missing values     : {total_missing}')
print(f'[2] Duplicate rows     : {dup_rows}')
print(f'[3] Duplicate Id vals  : {dup_ids}')
print(f'[4] Constant columns   : {len(const_cols)} → {const_cols}')
print(f'[5] Near-constant cols : {len(near_const_cols)} (≥99% same value)')
print(f'\nRaw shape              : {df.shape}')

---
## 2. TRANSFORM — Clean & Engineer Features

| Step | Action |
|------|--------|
| 1 | Drop fully constant columns |
| 2 | Drop near-constant columns (≥ 99 % same value) |
| 3 | Drop redundant `Total_sqr_footage` |
| 4 | Flag luxury outliers (`is_luxury`) |
| 5 | Flag financial-crisis-year sales (`is_crisis_year`) |
| 6 | Fix data types (year columns → int) |

In [ ]:
# ── Step 1: Drop fully constant columns (nunique ≤ 1) ───────────────────────
CONSTANT_COLS = [c for c in df.columns if df[c].nunique() <= 1]

df.drop(columns=CONSTANT_COLS, inplace=True)

print(f'✅  Step 1 — Dropped {len(CONSTANT_COLS)} constant column(s)')
print(f'   {CONSTANT_COLS}')
print(f'   Shape now : {df.shape}')

In [ ]:
# ── Step 2: Drop near-constant columns (≥ 99 % same value) ──────────────────
near_const_cols = [
    c for c in df.columns
    if df[c].value_counts(normalize=True).iloc[0] >= NEAR_CONST_THRESH
]

df.drop(columns=near_const_cols, inplace=True)

print(f'✅  Step 2 — Dropped {len(near_const_cols)} near-constant column(s) (≥99% same value)')
print(f'   Shape now : {df.shape}')

In [ ]:
# ── Step 3: Drop redundant engineered feature ────────────────────────────────
# Total_sqr_footage is 95 % correlated with TotalSF — keep TotalSF
if 'Total_sqr_footage' in df.columns:
    df.drop(columns=['Total_sqr_footage'], inplace=True)
    print("✅  Step 3 — Dropped 'Total_sqr_footage' (r=0.95 with TotalSF)")
else:
    print("ℹ️   Step 3 — 'Total_sqr_footage' not present, skipped")

print(f'   Shape now : {df.shape}')

In [ ]:
# ── Step 4: Flag luxury outliers ─────────────────────────────────────────────
# Legitimate high-quality homes in the top 1 % — NOT dropped, just flagged
luxury_threshold = df['Saleprice'].quantile(LUXURY_PERCENTILE)
df['is_luxury']  = (df['Saleprice'] > luxury_threshold).astype(int)

print(f'✅  Step 4 — Flagged {df["is_luxury"].sum()} luxury outlier(s) as is_luxury=1')
print(f'   Threshold : ${luxury_threshold:,.0f}')
print(f'   These are legitimate high-quality homes (OverallQual 8–10) — NOT dropped')
print(f'   Shape now : {df.shape}')

# Show the luxury homes
df.loc[df['is_luxury'] == 1, ['Id', 'OverallQual', 'Saleprice']].sort_values('Saleprice')

In [ ]:
# ── Step 5: Flag financial-crisis-year sales ─────────────────────────────────
crisis_ohe_cols = [f'YrSold_{y}' for y in CRISIS_YEARS if f'YrSold_{y}' in df.columns]

if crisis_ohe_cols:
    df['is_crisis_year'] = df[crisis_ohe_cols].max(axis=1).astype(int)
    src = 'OHE YrSold columns'
elif 'YrSold' in df.columns:
    df['is_crisis_year'] = df['YrSold'].isin(CRISIS_YEARS).astype(int)
    src = 'YrSold column'
else:
    print('⚠️  No YrSold column found — is_crisis_year not added')
    src = None

if src:
    print(f'✅  Step 5 — Flagged {df["is_crisis_year"].sum():,} crisis-year record(s) as is_crisis_year=1')
    print(f'   Source    : {src}')
    print(f'   Shape now : {df.shape}')

In [ ]:
# ── Step 6: Fix data types — year columns → int ──────────────────────────────
year_cols = [c for c in df.columns if 'Year' in c or 'YrBlt' in c]
for col in year_cols:
    df[col] = df[col].astype(int)

print(f'✅  Step 6 — Consolidated {len(year_cols)} year-column(s) to int dtype')
print(f'   Columns : {year_cols}')

In [ ]:
# ── Transform summary ─────────────────────────────────────────────────────────
print('=' * 55)
print('TRANSFORM SUMMARY')
print('=' * 55)
print(f'  Final shape          : {df.shape}')
print(f'  Rows                 : {df.shape[0]:,}')
print(f'  Columns              : {df.shape[1]}')
print(f'  Luxury homes flagged : {df["is_luxury"].sum()}')
print(f'  Crisis-yr flagged    : {df["is_crisis_year"].sum():,}')

df.head()

---
## 3. LOAD — Export Processed Datasets

In [ ]:
# ── Load 1: Save fully cleaned dataset ───────────────────────────────────────
os.makedirs(os.path.dirname(CLEAN_PATH), exist_ok=True)
df.to_csv(CLEAN_PATH, index=False)

size_kb = os.path.getsize(CLEAN_PATH) / 1024
print(f'✅  cleaned_data.csv saved')
print(f'   Path  : {CLEAN_PATH}')
print(f'   Shape : {df.shape}')
print(f'   Size  : {size_kb:.1f} KB')

In [ ]:
# ── Load 2: Build and save Tableau-ready dataset ──────────────────────────────
df_tableau = df.copy()
df_tableau['PriceCategory'] = pd.qcut(
    df_tableau['Saleprice'],
    q=4,
    labels=['Low', 'Medium', 'High', 'Very High']
)

df_tableau.to_csv(TABLEAU_PATH, index=False)

size_kb_tab = os.path.getsize(TABLEAU_PATH) / 1024
print(f'✅  tableau_ready.csv saved')
print(f'   Path  : {TABLEAU_PATH}')
print(f'   Shape : {df_tableau.shape}')
print(f'   Size  : {size_kb_tab:.1f} KB')

In [ ]:
# ── Final load summary ────────────────────────────────────────────────────────
print('=' * 55)
print('LOAD SUMMARY')
print('=' * 55)
print(f'  Records written     : {len(df):,}')
print(f'  Columns (cleaned)   : {df.shape[1]}')
print(f'  Columns (tableau)   : {df_tableau.shape[1]}')
print()
print('  PriceCategory distribution:')

price_dist = df_tableau['PriceCategory'].value_counts().sort_index()
for cat, cnt in price_dist.items():
    bar = '█' * int(cnt / 20)
    print(f'    {cat:<10} : {cnt:>4}  {bar}')